In [ ]:
# Run first. Anchor the working directory to the project root so that
# relative paths like "data/processed/..." resolve no matter where the
# notebook is launched from (notebooks live in notebooks/, data at root).
# Idempotent: re-running keeps you at the root.
import os
from pathlib import Path

_root = Path.cwd()
while not (_root / "CLAUDE.md").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
print("Working directory:", Path.cwd())

In [4]:
from datasets import load_dataset
import pandas as pd
import numpy as np

In [2]:
meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Grocery_and_Gourmet_Food",
    split="full",
    trust_remote_code=True,
)
meta.to_pandas().to_parquet("data/raw/grocery_meta.parquet")
print("saved")

raw/meta_categories/meta_Grocery_and_Gou(…): reconstructing file:   0%|          |  0.00B / 1.38GB            

raw/meta_categories/meta_Grocery_and_Gou(…): downloading bytes:           |  0.00B            

Generating full split: 0 examples [00:00, ? examples/s]

saved


In [3]:
gmetadata = pd.read_parquet("data/raw/grocery_meta.parquet")

print(gmetadata.shape)
print(gmetadata.columns.tolist())
print()
print("price coverage:", gmetadata["price"].notna().mean())
print()
print(gmetadata[["parent_asin", "title", "price", "average_rating", "rating_number"]].head())
print()
print(gmetadata["categories"].head(10).tolist())

(603274, 16)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

price coverage: 1.0

  parent_asin                                              title  price  \
0  B00NE08WM6                             Dark Roast Pure Coffee   None   
1  B084Q13Q5Q  PICARAS Galletas Peruanas Bañadas en Chocolate...  15.99   
2  B00KBRUYVM  Chipped Beef and Gravy By Patterson's - Great ...   None   
3  B0BN4PW255  Asher's Sugar Free Milk Chocolate Cordial Cher...  29.99   
4  B06X9DC27H              Messmer Peppermint 25 bags (6er pack)  29.99   

   average_rating  rating_number  
0             4.7              9  
1             4.5             12  
2             3.2              5  
3             5.0              6  
4             3.5              5  

[array(['Grocery & Gourmet Food', 'Beverages', 'Coffee'], dtype=object), array(['Grocery & Gour

In [8]:
meta = pd.read_parquet("data/raw/grocery_meta.parquet")

meta["price_num"] = pd.to_numeric(meta["price"], errors="coerce")
print("real price coverage:", meta["price_num"].notna().mean().round(3))
print()

meta["cat_l2"] = meta["categories"].apply(lambda c: c[1] if len(c) > 1 else None)
meta["cat_l3"] = meta["categories"].apply(lambda c: c[2] if len(c) > 2 else None)

print("products with no category:", meta["cat_l2"].isna().sum())
print()
print(meta["cat_l2"].value_counts().head(15))
print()
print(meta["cat_l3"].value_counts().head(20))

real price coverage: 0.372

products with no category: 68198

cat_l2
Pantry Staples                            210135
Snacks & Sweets                           109105
Beverages                                  95811
Breads & Bakery                            25033
Dairy, Eggs & Plant-Based Alternatives     11951
Produce                                    11777
Food & Beverage Gifts                      11444
Frozen                                      9185
Meat & Seafood                              9179
Deli & Prepared Foods                       7048
Breakfast Foods                             6662
Fresh Flowers & Live Indoor Plants          4273
Canned, Packaged & Baking                   3976
Breakfast Cereal                            3534
Alcoholic Beverages                         2682
Name: count, dtype: int64

cat_l3
Cooking & Baking                          95156
Snack Foods                               43241
Candy & Chocolate                         42436
Tea               

In [5]:
grocery = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Grocery_and_Gourmet_Food",
    split="full",
    trust_remote_code=True,
)
grocery.to_pandas().to_parquet("data/raw/grocery_reviews.parquet")
print("saved")

saved


In [9]:
reviews = pd.read_parquet(
    "data/raw/grocery_reviews.parquet",
    columns=["parent_asin", "rating"]
)

r = reviews.merge(
    meta[["parent_asin", "cat_l2", "cat_l3"]],
    on="parent_asin",
    how="left"
)

per_product = (
    r.groupby(["cat_l3", "parent_asin"])
     .size()
     .reset_index(name="n_reviews")
)

summary = per_product.groupby("cat_l3").agg(
    total_reviews=("n_reviews", "sum"),
    n_products=("parent_asin", "count"),
    median_reviews=("n_reviews", "median"),
    products_50plus=("n_reviews", lambda s: (s >= 50).sum()),
)

print(summary.sort_values("products_50plus", ascending=False).head(20))

                                        total_reviews  n_products  \
cat_l3                                                              
Cooking & Baking                              2280626       95132   
Snack Foods                                   1301619       43235   
Candy & Chocolate                             1114653       42432   
Herbs, Spices & Seasonings                     849825       33847   
Coffee                                        1423578       27297   
Tea                                            967184       35400   
Bottled Beverages, Water & Drink Mixes         861219       25955   
Sauces, Gravies & Marinades                    424797       20333   
Canned, Jarred & Packaged Foods                408455       17731   
Chocolate Candy                                385888       20598   
Cookies                                        338069       13245   
Soups, Stocks & Broths                         219662        7841   
Condiments & Salad Dressings      

In [10]:
coffee_asins = meta.loc[meta["cat_l3"] == "Coffee", "parent_asin"]

reviews_full = pd.read_parquet("data/raw/grocery_reviews.parquet")
coffee = reviews_full[reviews_full["parent_asin"].isin(coffee_asins)].copy()

coffee.to_parquet("data/interim/coffee_reviews.parquet")

print("reviews:", len(coffee))
print("products:", coffee["parent_asin"].nunique())
print("users:", coffee["user_id"].nunique())

reviews: 1423578
products: 27297
users: 1049484


In [11]:
coffee = pd.read_parquet("data/interim/coffee_reviews.parquet")
meta = pd.read_parquet("data/raw/grocery_meta.parquet")
meta["price_num"] = pd.to_numeric(meta["price"], errors="coerce")

# per-product stats from the reviews themselves
prod = coffee.groupby("parent_asin").agg(
    n_reviews=("rating", "size"),
    mean_rating=("rating", "mean"),
    neg_rate=("rating", lambda r: (r <= 2).mean()),
).reset_index()

# attach catalog info
prod = prod.merge(
    meta[["parent_asin", "title", "store", "price_num", "rating_number"]],
    on="parent_asin", how="left"
)

# competitive set: enough reviews to characterize
cset = prod[prod["n_reviews"] >= 50].copy()

print("products in competitive set:", len(cset))
print("reviews covered:", cset["n_reviews"].sum())
print("price coverage in set:", cset["price_num"].notna().mean().round(3))
print()
print("negative-rate distribution:")
print(cset["neg_rate"].describe().round(3))
print()
print("top brands by review volume:")
print(cset.groupby("store")["n_reviews"].sum().sort_values(ascending=False).head(15))

products in competitive set: 3273
reviews covered: 1237303
price coverage in set: 0.568

negative-rate distribution:
count    3273.000
mean        0.150
std         0.098
min         0.000
25%         0.082
50%         0.127
75%         0.191
max         0.952
Name: neg_rate, dtype: float64

top brands by review volume:
store
Starbucks                         50070
Green Mountain Coffee Roasters    48308
Death Wish Coffee Co.             45656
SAN FRANCISCO BAY                 45229
Lavazza                           35498
Crazy Cups                        34144
Nespresso                         31676
Solimo                            28296
Victor Allen                      23669
MAXWELL HOUSE                     22419
Folgers                           21597
Amazon Fresh                      18211
Peet's Coffee                     17576
Cafe Don Pablo                    16773
MAUD'S                            16560
Name: n_reviews, dtype: int64


In [12]:
cset.to_parquet("data/processed/coffee_competitive_set.parquet")
coffee[coffee["parent_asin"].isin(cset["parent_asin"])].to_parquet(
    "data/processed/coffee_reviews_cset.parquet"
)
print("saved")

saved
